# 📓 Semana 10 · Dia 1 — MLflow: experiments, runs e tracking

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | MLP, MLA |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Primeiro experimento com autologging |

---


## 📖 Teoria — O que é o MLflow

O **MLflow** é a plataforma open-source de ciclo de vida de ML — e está embutido no Databricks. Quatro componentes:

1. **Tracking**: registrar params, metrics, artifacts e código de cada run
2. **Models**: empacotar e versionar modelos (registry)
3. **Model Registry**: governança de modelos (aliases, stages)
4. **Evaluate/Tracing**: avaliar e debugar (LLMs também)

**Experiment** = projeto (coleção de runs). **Run** = uma execução de treino com params/métricas/artifacts. Sem MLflow, você perde rastreabilidade.


## 📖 Teoria — Autologging

O `mlflow.autolog()` captura **automaticamente** params, métricas e modelos de frameworks (sklearn, xgboost, pytorch...) — sem instrumentação manual. É o padrão recomendado.


### 💻 Na prática — Primeiro experimento

Treine um modelo simples de previsão de receita com autologging.


In [ ]:
# Preparar dados de treino a partir do Ouro
from pyspark.sql.functions import dayofweek, month, year
df = (spark.table("workspace.ouro.vendas_por_dia")
    .withColumn("dia_semana", dayofweek("data_venda"))
    .withColumn("mes", month("data_venda"))
    .withColumn("ano", year("data_venda"))
    .toPandas())
df.head()

In [ ]:
# Treino com autologging
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

mlflow.autolog()
X = df[["dia_semana", "mes", "ano"]]
y = df["receita_total"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="rf_vendas_v1"):
    modelo = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
    modelo.fit(X_train, y_train)
    print("Run registrado no experimento!")

In [ ]:
# Ver os runs registrados
experiments = mlflow.search_experiments()
for e in experiments:
    print(f"Experiment: {e.name} (id={e.experiment_id})")

### 💻 Na prática — Explorando na UI

1. **Experiments** (sidebar) → experimento do notebook.
2. Veja: params (n_estimators, max_depth), métricas (rmse, r2), artifacts (model, requirements).
3. Compare 2 runs com **Compare**.


> 🎯 **Dica de prova**: MLP/MLA cobram: experimento vs run, autolog, e como registrar params/métricas/artifacts. Pergunta clássica: 'o que o autolog registra?' → params, métricas, modelo e código da run.


## 🎯 Exercícios de fixação

**1.** Rode o treino 2x com n_estimators diferentes e compare na UI.

**2.** Registre uma métrica manual com mlflow.log_metric.

**3.** O que é um artifact? Dê 3 exemplos.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Comparar runs

Rode v1 e v2; em Experiments, selecione as duas e Compare: veja rmse/r2 lado a lado.

**2.** Métrica manual

```python
with mlflow.start_run():
    mlflow.log_metric('meu_rmse', 0.42)
```

**3.** Artifacts

Arquivos anexados à run: modelo serializado, gráficos, dataset de amostra, requirements.txt.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*